In [18]:
%pip install transformers torch -q

In [2]:
"""
Step 1: Environment + base model sanity check.
Run this in Colab (free GPU) or locally. Confirms the pretrained
Hindi GPT-2 model loads and can generate coherent-ish Hindi text
before we touch fine-tuning.
"""
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "surajp/gpt2-hindi"  # GPT-2 architecture, 88M params, pretrained from scratch on Hindi

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Using device:", device)

def complete_sentence(prompt, max_new_tokens=30, top_k=50, top_p=0.95, temperature=0.8):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=top_k,
            top_p=top_p,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

def next_word_candidates(prompt, k=5):
    """Single forward pass -> top-k next token candidates with probabilities."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]  # logits for the next token
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k)
    return [(tokenizer.decode([tid]), prob.item()) for tid, prob in zip(top_ids, top_probs)]

if __name__ == "__main__":
    test_prompts = ["भारत एक", "मुझे आज बहुत", "स्कूल जाना"]

    for p in test_prompts:
        print("\n--- Prompt:", p, "---")
        print("Completion:", complete_sentence(p))
        print("Top next-word candidates:")
        for word, prob in next_word_candidates(p):
            print(f"  {word!r}  ({prob:.3f})")

    print("\nVocab size:", tokenizer.vocab_size)
    print("Model params:", sum(p.numel() for p in model.parameters()))

Loading surajp/gpt2-hindi ...


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 50256), got 50257. This may result in unexpected behavior.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: surajp/gpt2-hindi
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda

--- Prompt: भारत एक ---
Completion: भारत एक बार फिर से नागपुर के भूतपूर्व मुख्यमंत्
Top next-word candidates:
  ' ऐस'  (0.100)
  ' व'  (0.082)
  ' ब'  (0.068)
  ' स'  (0.068)
  ' प'  (0.045)

--- Prompt: मुझे आज बहुत ---
Completion: मुझे आज बहुत चाहती हूँ , लेकिन हमारे समाज में जब भी किसी प्
Top next-word candidates:
  ' अच'  (0.154)
  ' ख'  (0.084)
  ' द'  (0.070)
  ' प'  (0.052)
  ' मज'  (0.045)

--- Prompt: स्कूल जाना ---
Completion: स्कूल जाना था।
में अपने जीवन की सार्थकता का हिस्सा होता
Top next-word candidates:
  ' पड'  (0.291)
  ' ह'  (0.268)
  ' थ'  (0.104)
  ' च'  (0.086)
  ' और'  (0.020)

Vocab size: 50257
Model params: 81912576


In [3]:
!pip install datasets -q

In [4]:
"""
Step 2 (v2): Build the fine-tuning corpus from OpenSubtitles Hindi data.
Downloads the raw Moses-format en-hi bitext directly from OPUS
(bypasses the HF `datasets` script-loading issue entirely) and
keeps only the Hindi side for our conversational corpus.
"""
import urllib.request
import zipfile
import re
import os

URL = "https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/moses/en-hi.txt.zip"
ZIP_PATH = "en-hi.txt.zip"

if not os.path.exists(ZIP_PATH):
    print("Downloading OpenSubtitles en-hi (moses format) from OPUS...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Download complete.")
else:
    print("Zip already downloaded, skipping.")

print("Extracting...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Files in archive:", z.namelist())
    z.extractall(".")

# The Hindi-side file is typically named OpenSubtitles.en-hi.hi
hi_file_candidates = [f for f in os.listdir(".") if f.endswith(".hi")]
if not hi_file_candidates:
    raise FileNotFoundError("Couldn't find the .hi file after extraction — check the printed file list above and adjust the filename.")
hi_file = hi_file_candidates[0]
print("Using Hindi file:", hi_file)

def clean_line(text):
    text = text.strip()
    if len(text) < 3 or len(text) > 200:
        return None
    text = re.sub(r"^[-–—]\s*", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = text.strip()
    if not text:
        return None
    return text

lines = []
with open(hi_file, "r", encoding="utf-8") as f:
    for raw_line in f:
        cleaned = clean_line(raw_line)
        if cleaned:
            lines.append(cleaned)

print(f"Total cleaned lines: {len(lines)}")

lines = list(dict.fromkeys(lines))  # dedupe, preserve order
print(f"After dedup: {len(lines)}")

with open("hindi_conversational_corpus.txt", "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line + "\n")

print("Saved to hindi_conversational_corpus.txt")
print("\nSample lines:")
for l in lines[:10]:
    print(" -", l)

Zip already downloaded, skipping.
Extracting...
Files in archive: ['OpenSubtitles.en-hi.en', 'OpenSubtitles.en-hi.hi', 'OpenSubtitles.en-hi.ids', 'README']
Using Hindi file: OpenSubtitles.en-hi.hi
Total cleaned lines: 92153
After dedup: 76750
Saved to hindi_conversational_corpus.txt

Sample lines:
 - साइकिल चोर
 - रिच्ची?
 - रिच्ची है क्या?
 - तुम बहरे हो क्या?
 - उठो! चलो.
 - और क्योंकि मैं राजमिस्त्री हूँ मुझे भूख से मर जाना चाहिए?
 - तुम मुझसे क्या चाहते हो?
 - बस धैर्य रखो.
 - हम देखेंगे कि हम क्या कर सकते हैं.
 - हम कुछ खोजने की कोशिश करेंगे.


In [5]:
"""
Step 3: Tokenize and chunk the corpus for causal LM fine-tuning.
Uses the SAME tokenizer as the base model (don't retrain the
tokenizer -- that would break the pretrained embeddings' meaning).
Concatenates all lines and splits into fixed-length blocks, which
is the standard approach for causal LM fine-tuning (as opposed to
padding each short line individually, which wastes compute).
"""
from transformers import AutoTokenizer
from datasets import Dataset
import os

MODEL_NAME = "surajp/gpt2-hindi"
CORPUS_PATH = "hindi_conversational_corpus.txt"
BLOCK_SIZE = 128  # tokens per training example; 128 is a reasonable size for short conversational lines

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading corpus...")
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Loaded {len(lines)} lines.")

# Join lines with EOS token so the model learns sentence boundaries
print("Tokenizing...")
full_text = tokenizer.eos_token.join(lines)
tokenized = tokenizer(full_text, return_tensors=None)["input_ids"]
print(f"Total tokens: {len(tokenized)}")

# Chunk into fixed-length blocks
def chunk_list(lst, size):
    for i in range(0, len(lst) - size + 1, size):
        yield lst[i:i + size]

blocks = list(chunk_list(tokenized, BLOCK_SIZE))
print(f"Total training blocks of size {BLOCK_SIZE}: {len(blocks)}")

# Build HF Dataset: input_ids == labels for causal LM (next-token prediction)
# attention_mask is all 1s since these are fixed-length blocks with no padding
attention_masks = [[1] * BLOCK_SIZE for _ in blocks]
dataset_dict = {"input_ids": blocks, "labels": [b[:] for b in blocks], "attention_mask": attention_masks}
hf_dataset = Dataset.from_dict(dataset_dict)

# 95/5 train/val split for perplexity tracking during training
split_dataset = hf_dataset.train_test_split(test_size=0.05, seed=42)
print(split_dataset)

split_dataset.save_to_disk("hindi_lm_dataset")
print("Saved tokenized dataset to ./hindi_lm_dataset")

print("\nSanity check - decoding first training block:")
print(tokenizer.decode(split_dataset["train"][0]["input_ids"]))

Loading corpus...
Loaded 76750 lines.
Tokenizing...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1612623 > 512). Running this sequence through the model will result in indexing errors


Total tokens: 1612623
Total training blocks of size 128: 12598
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 11968
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 630
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/11968 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/630 [00:00<?, ? examples/s]

Saved tokenized dataset to ./hindi_lm_dataset

Sanity check - decoding first training block:
ीं है.<|endoftext|>तुम एक प्यारी सवारी है वहाँ. / इसे मत छूना.<|endoftext|>इसे भी मत देखना नहीं है.<|endoftext|>बाहर निकल जाओ.<|endoftext|>या तो मुझ पर क्या देख नहीं है.<|endoftext|>हाँ,तुमबेहतरपरचलते हैं.<|endoftext|>वह वास्तव में अजीब किस्म का है. / हाँ, वह अच्छी तरह से मतलब है.<|endoftext|>मैंसा


In [6]:
"""
Step 4: Fine-tune surajp/gpt2-hindi on the conversational corpus.
Uses HF Trainer with causal LM loss. Tracks eval loss -> perplexity
each epoch so we can see if it's actually learning vs overfitting.
"""
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, default_data_collator
)
from datasets import load_from_disk
import torch
import math

MODEL_NAME = "surajp/gpt2-hindi"
DATASET_PATH = "hindi_lm_dataset"
OUTPUT_DIR = "hindi_gpt2_finetuned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# The tokenizer's EOS token was added as a special token at id 50257 --
# one slot past the base embedding matrix's size (rows 0-50256 only).
# Setting pad_token_id to that id (or using EOS at all) is an out-of-bounds
# embedding lookup unless we resize the embedding matrix to match the
# tokenizer's true length first. This is the actual fix -- our earlier
# attempt (just reassigning pad_token_id) pointed at the same broken id.
print("Tokenizer length:", len(tokenizer), "| Model embedding rows before resize:", model.get_input_embeddings().weight.shape[0])
model.resize_token_embeddings(len(tokenizer))
print("Model embedding rows after resize:", model.get_input_embeddings().weight.shape[0])

model.config.pad_token_id = tokenizer.eos_token_id
print("model.config.pad_token_id ->", model.config.pad_token_id, "(now a valid row)")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

dataset = load_from_disk(DATASET_PATH)
# input_ids, labels, and attention_mask are all present from Step 3.
# default_data_collator converts these python lists to tensors directly --
# no .set_format(type="torch") needed (that path triggers a datasets/
# torchvision version conflict in current Colab environments).

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=default_data_collator,
)

print("Starting fine-tuning...")
train_result = trainer.train()

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print(f"\nFinal eval loss: {eval_results['eval_loss']:.4f}")
print(f"Final perplexity: {perplexity:.2f}")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel saved to ./{OUTPUT_DIR}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: surajp/gpt2-hindi
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer length: 50258 | Model embedding rows before resize: 50257


[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model embedding rows after resize: 50258
model.config.pad_token_id -> 50257 (now a valid row)
Training on: cuda
Starting fine-tuning...


...

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training Loss,Validation Loss,Epoch
1.808158,1.890618,3



Final eval loss: 1.8906
Final perplexity: 6.62


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./hindi_gpt2_finetuned


In [7]:
"""
Step 4b: Compare base vs fine-tuned model on the same prompts.
Confirms fine-tuning actually improved things before we build on top of it.
"""
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

def load(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path).to(device)
    model.eval()
    return tok, model

print("Loading base model...")
base_tok, base_model = load("surajp/gpt2-hindi")
print("Loading fine-tuned model...")
ft_tok, ft_model = load("hindi_gpt2_finetuned")

def generate(tok, model, prompt, max_new_tokens=25):
    input_ids = tok.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(output[0], skip_special_tokens=True)

test_prompts = ["भारत एक", "मुझे आज बहुत", "स्कूल जाना", "तुम क्या", "हम कल"]

for p in test_prompts:
    print(f"\n--- Prompt: {p} ---")
    print("Base:      ", generate(base_tok, base_model, p))
    print("Fine-tuned:", generate(ft_tok, ft_model, p))

Loading base model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: surajp/gpt2-hindi
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading fine-tuned model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


--- Prompt: भारत एक ---
Base:       भारत एक स्वायत्त देश है।
मौसम विभाग के अधि
Fine-tuned: भारत एक विशाल देश है.मुझे लगता है कि मैं खु

--- Prompt: मुझे आज बहुत ---
Base:       मुझे आज बहुत बार आश्चर्य हुआ कि हमारे यहाँ भी अनेक व
Fine-tuned: मुझे आज बहुत याद आती है.आप मुझे याद दिलाना चाहत

--- Prompt: स्कूल जाना ---
Base:       स्कूल जाना चाहता है, तो मैंने कहा “प्रिंसिपल जी क
Fine-tuned: स्कूल जाना है, हम बारे में बात करते हैं, तुम एक ही लड़क

--- Prompt: तुम क्या ---
Base:       तुम क्या है ?
फोटोग्राफी और सिद्धांत में ग
Fine-tuned: तुम क्या कर रहे हो?कोई प्रेज़ीडेंट को नियुक

--- Prompt: हम कल ---
Base:       हम कल्पना की जा सकती है कि किसी भी क्रिया
Fine-tuned: हम कल्पना कर सकते हैं कि आप क्या करने जा रहे हैं.यह


In [13]:
"""
Step 5: The core predictor module.

Two functions, both built on the fine-tuned model:
1. predict_next_word()  -- returns top-k WHOLE-WORD candidates (not subword
   fragments -- generates token-by-token until a word boundary, as we
   flagged back in Step 1).
2. complete_sentence()  -- autoregressive generation that stops after
   roughly 2-3 sentences (counts sentence-ending punctuation), not a
   fixed token count, so short prompts don't get needlessly padded out
   and longer prompts don't get cut mid-thought.
"""
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = "hindi_gpt2_finetuned"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)
model.eval()

SENTENCE_END_CHARS = {".", "?", "!", "।"}  # । is the Hindi/Devanagari full stop (danda)
WORD_BOUNDARY_CHARS = {" ", "\n"} | SENTENCE_END_CHARS


def predict_next_word(prompt, k=5, max_token_steps=6):
    """
    Returns top-k whole-word candidates for what comes next.
    Generates greedily token-by-token per candidate branch until hitting
    a word boundary (space/punctuation), since the tokenizer works in
    subwords, not whole words.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    # Pull a wider pool than k, since some will get filtered out as
    # mid-word continuation pieces (no leading space) below.
    pool_size = min(k * 4, probs.shape[-1])
    top_probs, top_ids = torch.topk(probs, pool_size)

    candidates = []
    for first_id, first_prob in zip(top_ids, top_probs):
        if len(candidates) >= k:
            break
        first_piece_raw = tokenizer.decode([first_id.item()])
        # Skip continuation-only tokens (no leading space) -- these mean
        # the model is predicting a merge into the PRIOR word (e.g. "कल"
        # -> "कल्पना"), not a genuine new next word. Displayed alone
        # they're meaningless fragments like '्पना'.
        if not first_piece_raw.startswith(" "):
            continue

        current_ids = torch.cat([input_ids, first_id.view(1, 1)], dim=1)
        word_piece = first_piece_raw

        # Look ahead: keep extending as long as the NEXT predicted token
        # is a continuation piece (no leading space -- same word). Stop as
        # soon as the next token would start a new word or hit punctuation.
        # (We must check the NEXT token's boundary, not the current one --
        # the current/first token itself always starts with a leading space
        # by BPE convention, which is not itself a stopping signal.)
        steps = 1
        while steps < max_token_steps:
            with torch.no_grad():
                next_logits = model(current_ids).logits[0, -1, :]
            next_id = torch.argmax(next_logits).item()
            next_piece = tokenizer.decode([next_id])
            if (next_piece.startswith(" ")
                    or any(c in next_piece for c in SENTENCE_END_CHARS)
                    or next_id == tokenizer.eos_token_id):
                break  # next token starts a NEW word -- current word is complete
            word_piece += next_piece
            current_ids = torch.cat([current_ids, torch.tensor([[next_id]]).to(device)], dim=1)
            steps += 1

        word = word_piece.strip(" .?!।\n")
        if word:
            candidates.append((word, first_prob.item()))

    # Dedupe (different token paths can land on the same surface word)
    seen = set()
    deduped = []
    for word, prob in candidates:
        if word not in seen:
            seen.add(word)
            deduped.append((word, prob))
    return deduped


def complete_sentence(prompt, max_sentences=3, max_new_tokens=80, top_k=50, top_p=0.95, temperature=0.8):
    """
    Generates a completion capped at roughly max_sentences sentences,
    with max_new_tokens as a hard safety ceiling (in case punctuation
    never appears). Input length isn't force-padded -- a short prompt
    naturally tends to produce a short-ish completion from this model.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids

    sentence_count = 0
    new_tokens = 0

    with torch.no_grad():
        while new_tokens < max_new_tokens and sentence_count < max_sentences:
            logits = model(generated).logits[0, -1, :]
            logits = logits / temperature

            # top-p (nucleus) sampling
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = cumulative_probs > top_p
            cutoff[1:] = cutoff[:-1].clone()
            cutoff[0] = False
            sorted_logits[cutoff] = float("-inf")
            probs = torch.softmax(sorted_logits, dim=-1)
            next_token_sorted_idx = torch.multinomial(probs, 1)
            next_id = sorted_indices[next_token_sorted_idx]

            generated = torch.cat([generated, next_id.view(1, 1)], dim=1)
            new_tokens += 1

            piece = tokenizer.decode([next_id.item()])
            if any(c in piece for c in SENTENCE_END_CHARS):
                sentence_count += 1

            if next_id.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated[0], skip_special_tokens=True)


if __name__ == "__main__":
    test_prompts = ["भारत एक", "मुझे आज बहुत", "तुम क्या", "हम कल"]

    for p in test_prompts:
        print(f"\n=== Prompt: {p!r} ===")
        print("Next-word candidates:")
        for word, prob in predict_next_word(p, k=5):
            print(f"  {word!r}  ({prob:.3f})")
        print("Sentence completion (up to 3 sentences):")
        print(" ", complete_sentence(p, max_sentences=3))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


=== Prompt: 'भारत एक' ===
Next-word candidates:
  'संयुक्'  (0.088)
  'विशाल'  (0.078)
  'बेहतरीन'  (0.062)
  'प्राची'  (0.057)
  'राज्य'  (0.040)
Sentence completion (up to 3 sentences):
  भारत एक देश है।

=== Prompt: 'मुझे आज बहुत' ===
Next-word candidates:
  'याद'  (0.219)
  'देर'  (0.065)
  'खुशी'  (0.062)
  'प्यार'  (0.057)
  'ज्यादा'  (0.046)
Sentence completion (up to 3 sentences):
  मुझे आज बहुत याद आती है.

=== Prompt: 'तुम क्या' ===
Next-word candidates:
  'कर'  (0.346)
  'कह'  (0.079)
  'चाहते'  (0.063)
  'सोचते'  (0.051)
  'जानते'  (0.047)
Sentence completion (up to 3 sentences):
  तुम क्या कर रहे हैं?

=== Prompt: 'हम कल' ===
Next-word candidates:
  'रात'  (0.181)
  'के'  (0.109)
  'सुबह'  (0.048)
  'हमारे'  (0.046)
  'मिलेंग'  (0.034)
Sentence completion (up to 3 sentences):
  हम कल शाम को उसे नाम से नामित करेंगे.


In [17]:
"""
Step 6: Interactive CLI for the Hindi word predictor / sentence completer.
Matches the Sentence-Parser-NLP style: interactive loop, pick a mode,
run it, loop back until the user quits.
"""

def print_banner():
    print("=" * 60)
    print("       HINDI WORD PREDICTOR / SENTENCE COMPLETER")
    print("=" * 60)

def print_next_word_results(prompt, candidates):
    print(f"\nNext-word predictions for: '{prompt}'")
    if not candidates:
        print("  (no confident whole-word candidates found)")
        return
    for i, (word, prob) in enumerate(candidates, 1):
        print(f"  {i}. {word}  ({prob:.3f})")

def print_completion_result(prompt, completion):
    print(f"\nCompletion for: '{prompt}'")
    print(f"  {completion}")

def main():
    print_banner()
    while True:
        prompt = input("\nEnter a partial Hindi sentence (or 'q' to quit): ").strip()
        if prompt.lower() == "q":
            print("Goodbye.")
            break
        if not prompt:
            print("Please enter some text.")
            continue

        print("\nChoose mode:")
        print("1. Predict next word")
        print("2. Complete the sentence")
        print("q. Quit")
        mode = input("> ").strip()

        if mode == "q":
            print("Goodbye.")
            break
        elif mode == "1":
            candidates = predict_next_word(prompt, k=5)
            print_next_word_results(prompt, candidates)
        elif mode == "2":
            completion = complete_sentence(prompt, max_sentences=3)
            print_completion_result(prompt, completion)
        else:
            print("Invalid choice, try again.")

if __name__ == "__main__":
    main()

       HINDI WORD PREDICTOR / SENTENCE COMPLETER

Enter a partial Hindi sentence (or 'q' to quit): हम कल्पना कर सकते हैं

Choose mode:
1. Predict next word
2. Complete the sentence
q. Quit
> 1

Next-word predictions for: 'हम कल्पना कर सकते हैं'
  1. कि  (0.513)
  2. जब  (0.037)
  3. आप  (0.024)
  4. तुम्हें  (0.023)
  5. और  (0.020)

Enter a partial Hindi sentence (or 'q' to quit): हम कल्पना कर सकते हैं

Choose mode:
1. Predict next word
2. Complete the sentence
q. Quit
> 2

Completion for: 'हम कल्पना कर सकते हैं'
  हम कल्पना कर सकते हैं कि मैं सब कुछ किया.

Enter a partial Hindi sentence (or 'q' to quit): Jab mai english mein likhta hun

Choose mode:
1. Predict next word
2. Complete the sentence
q. Quit
> 2

Completion for: 'Jab mai english mein likhta hun'
  Jab mai english mein likhta hun,

Enter a partial Hindi sentence (or 'q' to quit): q
Goodbye.
